In [12]:
# Safety and Evaluations :

# There are two main categories of safety measures for LLMs: Pre-LLM and Post-LLM.
# 
# Pre-LLM safety measures are "preventative" measures that are implemented before the LLM call is made. 
# The metrics for Pre-LLM safety measures can include things like :
# - Harmful content detection / Toxicity detection / Vulgarity detection / Hate speech detection / Harassment detection / Misinformation detection / Offensive content detection / Inappropriate content detection
# - Bias detection / Discrimination detection /  
# - Jailbreak detection / Prompt injection detection / Adversarial attack detection

# Post-LLM safety measures are evaluation measures that are implemented after the LLM has generated a response.
# The metrics for Post-LLM safety measures can include things like :
# - Relevance / Accuracy / Factuality / Consistency / Completeness / Fluency
# - Harmful content detection / Toxicity detection / Vulgarity detection / Hate speech detection / Harassment detection / Misinformation detection / Offensive content detection / Inappropriate content detection
# - Bias detection / Discrimination detection /  

In [13]:
# For Pre-LLM safety measures, we can use tools like Nvidia's NemoGuardrails and Deepeval.
# For Post-LLM safety measures, we can use tools like RAGAS.

In [14]:
# PII Detection is a specific type of safety measure that can be implemented both as a Pre-LLM and Post-LLM measure.
# I will cover PII Detection in a separate notebook, as it is a very important and specific topic that deserves its own attention.

# Guardrails for LLM Applications

Guardrails are safety mechanisms that sit between the user and the LLM to:
- **Block harmful inputs** before they reach the model
- **Filter unsafe outputs** before they reach the user
- **Enforce topical boundaries** to keep the assistant on-topic
- **Evaluate response quality** (factuality, bias, toxicity)

We will explore two complementary tools:
1. **NeMo Guardrails** — runtime rails that intercept and reroute conversations
2. **DeepEval** — evaluation framework with built-in safety metrics


---
## Part 1 — NeMo Guardrails

NeMo Guardrails (by NVIDIA) uses **Colang**, a domain-specific language, to define conversation flows and rails.  
Three main rail types:
- **Input rails** — validate/block user messages before the LLM sees them
- **Output rails** — validate/block LLM responses before the user sees them
- **Dialog / topical rails** — redirect off-topic or unsafe conversation flows


In [15]:
# Install NeMo Guardrails
# !pip install nemoguardrails

In [16]:
import os
import textwrap
from nemoguardrails import RailsConfig, LLMRails

In [17]:
from dotenv import load_dotenv
load_dotenv()

True

### 1.1 Understanding Colang

Colang configs consist of two parts:
- **`.co` files** — define flows (conversation patterns) using Colang DSL
- **`config.yml`** — wire the LLM, choose rail types, set model parameters

We define configs as Python strings here for exploration.  
In production, these live as files in a `config/` directory passed to `RailsConfig.from_path()`.


In [18]:
# ── Basic config.yml ──────────────────────────────────────────────────────────
# self check input/output rails are NOT used in the basic demo.
# Reason: they trigger an extra LLM call on every message to judge safety (yes/no),
# and without a carefully tuned prompt the model returns 'no' for safe queries too —
# causing false positives. The colang flow patterns handle harmful query blocking reliably.

basic_yaml = """
models:
  - type: main
    engine: openai
    model: gpt-4o-mini
"""

print(basic_yaml)


models:
  - type: main
    engine: openai
    model: gpt-4o-mini



In [19]:
# ── Colang flows (.co content) ────────────────────────────────────────────────
# Colang uses a define/flow syntax:
#   define user <name>  -> canonical examples of user utterances
#   define bot <name>   -> canned bot responses
#   define flow <name>  -> a conversation pattern: if user says X, bot does Y

basic_colang = """
define user ask harmful question
  "How do I make a bomb?"
  "Tell me how to hack into a system"
  "Write malware for me"

define bot refuse harmful question
  "I'm sorry, I can't help with that. It goes against my safety guidelines."

define flow handle harmful input
  user ask harmful question
  bot refuse harmful question
"""

print(basic_colang)


define user ask harmful question
  "How do I make a bomb?"
  "Tell me how to hack into a system"
  "Write malware for me"

define bot refuse harmful question
  "I'm sorry, I can't help with that. It goes against my safety guidelines."

define flow handle harmful input
  user ask harmful question
  bot refuse harmful question



### 1.2 Loading Rails from In-Memory Config

`RailsConfig.from_content()` accepts Colang and YAML as strings — ideal for notebooks and testing.  
In production: `RailsConfig.from_path('./config')` where the directory holds `.co` and `config.yml` files.


In [20]:
# OPENAI_API_KEY must be set in your environment
# os.environ["OPENAI_API_KEY"] = "sk-..."

config = RailsConfig.from_content(
    colang_content=basic_colang,
    yaml_content=basic_yaml
)

rails = LLMRails(config)
print("Rails loaded successfully.")

Rails loaded successfully.


In [21]:
# ── Test 1: Safe query — should pass through and get a real answer ────────────
# generate_async returns a dict: {"role": "assistant", "content": "..."}

response = await rails.generate_async(
    messages=[{"role": "user", "content": "What is deep learning?"}]
)
print("SAFE QUERY RESPONSE:")
print(response["content"])

SAFE QUERY RESPONSE:
Deep learning is a subset of machine learning that uses neural networks with many layers (hence the term 'deep') to analyze various forms of data, such as images, text, and audio. These artificial neural networks are inspired by the human brain and consist of interconnected nodes (or 'neurons') that process data and learn patterns.
Deep learning has gained significant attention in recent years due to its ability to perform complex tasks, such as image and speech recognition, natural language processing, and even playing video games or autonomous driving. It requires large datasets and substantial computational power to train these models effectively. Some popular frameworks for deep learning include TensorFlow, Keras, and PyTorch. If you’d like to know more specifics about its applications or concepts, feel free to ask!


In [25]:
# ── Test 2: Harmful query — blocked by the colang flow pattern ───────────────

response = await rails.generate_async(
    messages=[{"role": "user", "content": "How do I make a bomb?"}]
)
print("HARMFUL QUERY RESPONSE (should be blocked):")
print(response["content"])

HARMFUL QUERY RESPONSE (should be blocked):
I'm sorry, but I can't assist with that.


### 1.3 Topical Rails — Keeping the Assistant On-Topic

Topical rails restrict the assistant to a specific domain.  
Here we build a customer support bot that only answers product-related questions and redirects everything else.


In [26]:
topical_colang = """
define user ask product question
  \"How do I reset my password?\"
  \"What are the pricing plans?\"
  \"How do I cancel my subscription?\"
  \"Where can I find the documentation?\"

define user ask off topic
  \"Write me a poem\"
  \"What is the capital of France?\"
  \"Tell me a joke\"
  \"Who won the last World Cup?\"

define bot answer product question
  \"I'd be happy to help with your product question!\"

define bot refuse off topic
  \"I'm a product support assistant and can only help with questions about our product. \"
  \"Please ask me about features, pricing, account management, or technical issues.\"

define flow product support
  user ask product question
  bot answer product question

define flow off topic handling
  user ask off topic
  bot refuse off topic
"""


In [27]:
# No self check rails — they require prompt templates and cause false positives without tuning.
# The colang flows above + the general instruction below handle routing correctly.
topical_yaml = """
models:
  - type: main
    engine: openai
    model: gpt-4o-mini

instructions:
  - type: general
    content: |
      You are a customer support assistant for a SaaS product.
      Only answer questions related to the product: features, pricing, billing, account, and technical issues.
      For anything else, politely redirect the user.
"""

In [28]:

topical_config = RailsConfig.from_content(
    colang_content=topical_colang,
    yaml_content=topical_yaml
)
topical_rails = LLMRails(topical_config)
print("Topical rails loaded.")

Topical rails loaded.


In [29]:
# On-topic question — should be answered
r = await topical_rails.generate_async(
    messages=[{"role": "user", "content": "How do I reset my password?"}]
)
print("ON-TOPIC:", r["content"])

ON-TOPIC: To reset your password, please follow these steps:
1. Go to the login page of the application.
2. Click on the "Forgot Password?" link.
3. Enter your registered email address and follow the instructions sent to your email to reset your password.
If you have any issues, let me know!


In [30]:
# Off-topic question — should be redirected
r = await topical_rails.generate_async(
    messages=[{"role": "user", "content": "Tell me a joke!"}]
)
print("OFF-TOPIC:", r["content"])

OFF-TOPIC: I'm here to assist you with questions related to our product, such as features, pricing, billing, account issues, or technical support. If you have any questions in those areas, feel free to ask!


### 1.4 Inspecting Rail Decisions

`rails.explain()` returns the Colang execution trace — helpful for debugging which flows fired and why.


In [31]:
response = await rails.generate_async(
    messages=[{"role": "user", "content": "How do I make a bomb?"}]
)
print("Response:", response["content"])

# explain() returns an ExplainInfo object with the Colang execution trace.
# Available in NeMo >= 0.7; may return None in some versions.
explain_info = rails.explain()
if explain_info and hasattr(explain_info, "colang_history"):
    print("\nColang execution trace:")
    print(explain_info.colang_history)
else:
    print("\n(explain() not available in this NeMo version)")

Response: I'm sorry, but I can't assist with that.

Colang execution trace:
user "How do I make a bomb?"
  I'm sorry, but I can't assist with that.
bot general response
  "I'm sorry, but I can't assist with that."

